# Загрузка данных

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from tqdm import tqdm

In [ ]:
df_m2_macro = pd.read_csv("data_for_models/df_m2_macro.csv", low_memory=False)

In [ ]:
df_m2_macro.info()

## CatBoost M2.2

In [ ]:
groups = df_m2_macro["region_name"]
results = []
gkf = GroupKFold(n_splits=5)

In [ ]:
models_config = {
    "M2.5.0_structure": {
        "cat": ["role_name", "schedule_id", "employment_id", "experience_ord"],
        "num": [],
    },
    "M2.5.1_macro_core": {
        "cat": ["role_name", "schedule_id", "employment_id", "experience_ord"],
        "num": ["GRP_K", "U_delta", "macro_k_u_imputed"],
    },
    "M2.5.2_full": {
        "cat": ["role_name", "schedule_id", "employment_id", "experience_ord"],
        "num": ["GRP_K", "U_delta", "macro_k_u_imputed", "RK"],
    },
}

In [ ]:
catboost_params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "RMSE",
    "task_type": "GPU",
    "devices": "0",
    "gpu_ram_part": 0.8,
    "random_seed": 42,
    "verbose": 100,
}

In [ ]:
N_FOLDS = int(gkf.n_splits)
for model_name, config in tqdm(models_config.items(), desc="Models", position=0):
    print(f"\nTraining {model_name}...")
    features = config["cat"] + config["num"]
    X = df_m2_macro[features]
    y = df_m2_macro["salary_from_log"]
    rmse_scores, mae_scores, r2_scores = [], [], []
    for fold_idx, (train_idx, val_idx) in enumerate(
        tqdm(
            gkf.split(X, y, groups),
            desc=f"Folds ({model_name})",
            position=1,
            leave=False,
        )
    ):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model = CatBoostRegressor(**catboost_params)
        model.fit(
            X_train,
            y_train,
            cat_features=config["cat"],
            eval_set=(X_val, y_val),
            verbose=0,
        )
        preds = model.predict(X_val)
        rmse_scores.append(np.sqrt(mean_squared_error(y_val, preds)))
        mae_scores.append(mean_absolute_error(y_val, preds))
        r2_scores.append(r2_score(y_val, preds))
    results.append(
        {
            "model": model_name,
            "CV_RMSE_mean": float(np.mean(rmse_scores)),
            "CV_RMSE_std": float(np.std(rmse_scores)),
            "CV_MAE_mean": float(np.mean(mae_scores)),
            "CV_MAE_std": float(np.std(mae_scores)),
            "CV_R2_mean": float(np.mean(r2_scores)),
            "CV_R2_std": float(np.std(r2_scores)),
            "n_folds": N_FOLDS,
        }
    )

In [ ]:
results_df_check = pd.DataFrame(results)
required_cols = {
    "model",
    "CV_RMSE_mean",
    "CV_RMSE_std",
    "CV_MAE_mean",
    "CV_MAE_std",
    "CV_R2_mean",
    "CV_R2_std",
    "n_folds",
}
missing_cols = required_cols - set(results_df_check.columns)
assert not missing_cols, f"В results отсутствуют колонки: {missing_cols}"
display(results_df_check)

## Анализ метрик

In [ ]:
results_df = pd.DataFrame(results).copy()
results_df_macro = results_df.sort_values("CV_RMSE_mean")
results_df_macro

In [ ]:
baseline_rmse = results_df.loc[
    results_df["model"] == "M2.5.0_structure", "CV_RMSE_mean"
].values[0]
baseline_r2 = results_df.loc[
    results_df["model"] == "M2.5.0_structure", "CV_R2_mean"
].values[0]
results_df["RMSE_improvement_%"] = (
    baseline_rmse - results_df["CV_RMSE_mean"]
) / baseline_rmse * 100
results_df["R2_gain"] = results_df["CV_R2_mean"] - baseline_r2
results_df = results_df.sort_values("model")
display(results_df.round(6))

In [ ]:
#Сохраним метрики
metrics_dir = Path("data_for_models")
metrics_dir.mkdir(parents=True, exist_ok=True)
m25_cv_metrics = results_df.copy()
preferred_cols = [
    "model",
    "CV_RMSE_mean",
    "CV_RMSE_std",
    "CV_MAE_mean",
    "CV_MAE_std",
    "CV_R2_mean",
    "CV_R2_std",
    "n_folds",
    "RMSE_improvement_%",
    "R2_gain",
]
m25_cv_metrics = m25_cv_metrics[[c for c in preferred_cols if c in m25_cv_metrics.columns]]
m25_cv_metrics = m25_cv_metrics.rename(columns={"model": "Model"})
m25_cv_metrics_path = metrics_dir / "m2_5_cv_metrics.csv"
m25_cv_metrics.to_csv(m25_cv_metrics_path, index=False, encoding="utf-8-sig")
print(f"Saved: {m25_cv_metrics_path}")
display(m25_cv_metrics.round(6))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df["model"], results_df["CV_R2_mean"], marker="o")
plt.title("M2.5: динамика CV R² при добавлении региона и макропризнаков")
plt.xlabel("Конфигурация")
plt.ylabel("CV_R2_mean")
plt.xticks(rotation=25, ha="right")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Выберем лучшую конфигурацию по CV_RMSE_mean (меньше = лучше)
best_row = results_df.loc[results_df["CV_RMSE_mean"].idxmin()]
best_model_name = best_row["model"]
best_config = models_config[best_model_name]
best_features = best_config["cat"] + best_config["num"]
print(f"Best model by CV_RMSE_mean: {best_model_name}")
display(best_row.to_frame().T)

In [ ]:
# Сделаем финальный refit выбранной лучшей конфигурации на полной выборке
X_final = df_m2_macro[best_features]
y_final = df_m2_macro["salary_from_log"]
best_model = CatBoostRegressor(**catboost_params)
best_model.fit(X_final, y_final, cat_features=best_config["cat"], verbose=0)
y_pred = best_model.predict(X_final)
rmse_best = np.sqrt(mean_squared_error(y_final, y_pred))
mae_best = mean_absolute_error(y_final, y_pred)
r2_best = r2_score(y_final, y_pred)
best_model_metrics = pd.DataFrame(
    [
        {
            "Model": f"{best_model_name} (best, in-sample)",
            "RMSE": rmse_best,
            "MAE": mae_best,
            "R2": r2_best,
        }
    ]
)
print(best_model_metrics.round(4))

## Feature Importance и SHAP

In [ ]:
#Строим график feature_importance
feature_importance = best_model.get_feature_importance(type="FeatureImportance")
fi_df = pd.DataFrame(
    {"feature": best_model.feature_names_, "importance": feature_importance}
).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(data=fi_df, x="importance", y="feature", palette="viridis")
plt.title(f"Feature Importance — {best_model_name}", fontsize=14)
plt.xlabel("Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
pool = Pool(X_final, cat_features=best_config["cat"])
shap_values = best_model.get_feature_importance(pool, type="ShapValues")
shap_values = shap_values[:, :-1]

In [ ]:
shap_df = pd.DataFrame(
    np.abs(shap_values).mean(axis=0),
    index=best_model.feature_names_,
    columns=["mean_abs_shap"],
).sort_values("mean_abs_shap", ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=shap_df.reset_index(), x="mean_abs_shap", y="index", palette="coolwarm")
plt.title(f"Mean |SHAP| — {best_model_name}", fontsize=14)
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
fi_df.to_csv("unload data/feature_importance_m21_4.csv", index=False)

In [ ]:
#Задаём группы признаков
structure_features = ["role_name", "schedule_id", "employment_id", "experience_ord"]
rk_features = ["RK"]
macro_features = ["GRP_K", "U_delta", "macro_k_u_imputed"]

In [ ]:
total_shap = shap_df["mean_abs_shap"].sum()

group_importance = {
    "Structure": shap_df.loc[
        shap_df.index.isin(structure_features), "mean_abs_shap"
    ].sum()
    / total_shap,
    "RK": shap_df.loc[
        shap_df.index.isin(rk_features), "mean_abs_shap"
    ].sum()
    / total_shap,
    "Macro": shap_df.loc[
        shap_df.index.isin(macro_features), "mean_abs_shap"
    ].sum()
    / total_shap,
}

group_df = pd.DataFrame.from_dict(
    group_importance, orient="index", columns=["share"]
).sort_values("share", ascending=False)
group_df

In [ ]:
plt.figure(figsize=(9, 5))
plot_data = group_df.reset_index().rename(columns={"index": "group"})
sns.barplot(data=plot_data, x="share", y="group", palette="Set2")
plt.title("Grouped SHAP — Structure / Region / Macro", fontsize=14)
plt.xlabel("Доля суммарного mean |SHAP|")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
colors = {"Structure": "#4C72B0", "RK": "#55A868", "Macro": "#C44E52"}
colors = {
    'Structure': '#4C72B0',
    'RK': '#55A868',
    'Macro': '#C44E52'
}
plt.figure(figsize=(8, 2))

left = 0

for group, value in group_importance.items():
    plt.barh(
        y='Model',
        width=value,
        left=left,
        color=colors[group],
        label=f"{group} ({value:.1%})"
    )
    left += value

plt.xlim(0, 1)
plt.title(f'Stacked SHAP Importance - {best_model_name}', fontsize=13)
plt.xlabel('Share of total SHAP')
plt.yticks([])
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)
plt.tight_layout()
plt.show()

# Сравним метрики моделей этапов M1, M2.1, M2.2

In [ ]:
m1_cv_metrics = pd.read_csv("data_for_models/m1_cv_metrics.csv")
m2_cv_metrics = pd.read_csv("data_for_models/m2_cv_metrics.csv")
m25_cv_metrics = pd.read_csv("data_for_models/m2_5_cv_metrics.csv")
# Нормализация имен столбцов моделей
if "model" in m2_cv_metrics.columns:
    m2_cv_metrics = m2_cv_metrics.rename(columns={"model": "Model"})
if "model" in m25_cv_metrics.columns:
    m25_cv_metrics = m25_cv_metrics.rename(columns={"model": "Model"})
# --- M1: OLS / Ridge (GroupKFold)
m1_part = m1_cv_metrics.loc[
    m1_cv_metrics["Model"].isin([
        "OLS (sklearn pipeline, GroupKFold)",
        "Ridge (sklearn pipeline, GroupKFold)"
    ])
].copy()
m1_part["Stage"] = m1_part["Model"].map({
    "OLS (sklearn pipeline, GroupKFold)": "M1.1",
    "Ridge (sklearn pipeline, GroupKFold)": "M1.2"
})
m1_part["Algorithm"] = m1_part["Stage"].map({
    "M1.1": "OLS",
    "M1.2": "Ridge"
})
m1_part["Features"] = "Structural"
# --- M2: лучший spatial
m2_part = m2_cv_metrics.loc[
    m2_cv_metrics["Model"].isin(["M2.1.4_distance"])
].copy()
m2_part["Stage"] = m2_part["Model"].map({
    "M2.1.4_distance": "M2.1.4"
})
m2_part["Algorithm"] = "CatBoost"
m2_part["Features"] = "Structural + Spatial"
# --- M2.5: baseline + full
m25_part = m25_cv_metrics.loc[
    m25_cv_metrics["Model"].isin(["M2.5.0_structure", "M2.5.2_full"])
].copy()
m25_part["Stage"] = m25_part["Model"].map({
    "M2.5.0_structure": "M2.5.0",
    "M2.5.2_full": "M2.5.2"
})
m25_part["Algorithm"] = "CatBoost"
m25_part["Features"] = m25_part["Stage"].map({
    "M2.5.0": "Structural",
    "M2.5.2": "Structural + Macro"
})
# Единая схема
common_cols = [
    "Stage", "Algorithm", "Features",
    "CV_RMSE_mean", "CV_RMSE_std",
    "CV_MAE_mean", "CV_MAE_std",
    "CV_R2_mean", "CV_R2_std",
    "n_folds"
]
m1_part = m1_part[common_cols].copy()
m2_part = m2_part[common_cols].copy()
m25_part = m25_part[common_cols].copy()
final_results_comp = pd.concat([m1_part, m2_part, m25_part], ignore_index=True)
stage_order = ["M1.1", "M1.2", "M2.1.4", "M2.5.0", "M2.5.2"]
final_results_comp["Stage"] = pd.Categorical(
    final_results_comp["Stage"],
    categories=stage_order,
    ordered=True
)
final_results_comp = final_results_comp.sort_values("Stage").reset_index(drop=True)
final_results_comp["CV_RMSE"] = final_results_comp["CV_RMSE_mean"]
final_results_comp["CV_R2"] = final_results_comp["CV_R2_mean"]
display(final_results_comp.round(6))

In [ ]:
sns.set(style="whitegrid", font_scale=1.1)
plot_df = final_results_comp.copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV RMSE
sns.barplot(
    data=plot_df,
    x="Stage",
    y="CV_RMSE_mean",
    hue="Features",
    dodge=False,
    palette="Blues",
    ax=axes[0]
)
axes[0].set_title("CV RMSE by Stage (M1 / M2 / M2.5)")
axes[0].set_xlabel("Model Stage")
axes[0].set_ylabel("CV_RMSE_mean")
axes[0].legend(title="Features", loc="lower right")

for p in axes[0].patches:
    h = p.get_height()
    if pd.notna(h):
        axes[0].annotate(
            f"{h:.4f}",
            (p.get_x() + p.get_width() / 2, h),
            ha="center",
            va="bottom",
            fontsize=9,
            xytext=(0, 3),
            textcoords="offset points"
        )

# CV R2
sns.barplot(
    data=plot_df,
    x="Stage",
    y="CV_R2_mean",
    hue="Features",
    dodge=False,
    palette="Greens",
    ax=axes[1]
)
axes[1].set_title("CV R2 by Stage (M1 / M2 / M2.5)")
axes[1].set_xlabel("Model Stage")
axes[1].set_ylabel("CV_R2_mean")
axes[1].legend(title="Features", loc="lower right")

for p in axes[1].patches:
    h = p.get_height()
    if pd.notna(h):
        axes[1].annotate(
            f"{h:.3f}",
            (p.get_x() + p.get_width() / 2, h),
            ha="center",
            va="bottom",
            fontsize=9,
            xytext=(0, 3),
            textcoords="offset points"
        )

plt.tight_layout()
plt.show()

In [ ]:
plot_df = final_results_comp.copy()
plot_df = plot_df.set_index("Stage").loc[
    ["M1.1", "M1.2", "M2.1.4", "M2.5.0", "M2.5.2"]
].reset_index()

stages = [
    f"{row.Stage}\n{row.Algorithm}\n{row.Features.lower()}"
    for row in plot_df.itertuples(index=False)
]
rmse = plot_df["CV_RMSE_mean"].tolist()
r2 = plot_df["CV_R2_mean"].tolist()

sns.set(style="whitegrid", context="talk")
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

# R²
axes[0].plot(stages, r2, marker="o", linewidth=2.5)
axes[0].set_title("Model performance improvement (CV R²)")
axes[0].set_xlabel("Model stage")
axes[0].set_ylabel("CV R² mean")
for i, v in enumerate(r2):
    axes[0].text(i, v + 0.002, f"{v:.3f}", ha="center")

# RMSE
axes[1].plot(stages, rmse, marker="o", linewidth=2.5, color="orange")
axes[1].set_title("Model performance improvement (CV RMSE)")
axes[1].set_xlabel("Model stage")
axes[1].set_ylabel("CV RMSE mean")
for i, v in enumerate(rmse):
    axes[1].text(i, v + 0.001, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

In [ ]:
# Приросты по отношению к двум референсам:

df_gain = final_results_comp.copy()


required_stages = {"M1.2", "M2.1.4", "M2.5.0", "M2.5.2"}
missing = required_stages - set(df_gain["Stage"].astype(str))
assert not missing, f"Не найдены этапы: {missing}"

# Базовые значения
rmse_m12 = float(df_gain.loc[df_gain["Stage"] == "M1.2", "CV_RMSE_mean"].iloc[0])
r2_m12   = float(df_gain.loc[df_gain["Stage"] == "M1.2", "CV_R2_mean"].iloc[0])

rmse_m214 = float(df_gain.loc[df_gain["Stage"] == "M2.1.4", "CV_RMSE_mean"].iloc[0])
r2_m214   = float(df_gain.loc[df_gain["Stage"] == "M2.1.4", "CV_R2_mean"].iloc[0])

# Расчёты
df_gain["RMSE_impr_vs_M1.2_%"] = (rmse_m12 - df_gain["CV_RMSE_mean"]) / rmse_m12 * 100
df_gain["R2_gain_vs_M1.2"]     = df_gain["CV_R2_mean"] - r2_m12

df_gain["RMSE_impr_vs_M2.1.4_%"] = (rmse_m214 - df_gain["CV_RMSE_mean"]) / rmse_m214 * 100
df_gain["R2_gain_vs_M2.1.4"]     = df_gain["CV_R2_mean"] - r2_m214

# Фокус на этапах, которые обычно интерпретируем в тексте
focus_order = ["M2.5.0", "M2.5.2"]
focus = (
    df_gain.set_index("Stage")
           .loc[focus_order, [
               "Algorithm", "Features",
               "CV_RMSE_mean", "CV_R2_mean",
               "RMSE_impr_vs_M1.2_%", "R2_gain_vs_M1.2",
               "RMSE_impr_vs_M2.1.4_%", "R2_gain_vs_M2.1.4"
           ]]
           .reset_index()
)

display(focus.round(6))